        # EWC 2026 Dota 2 fact-agent, clean version

        Этот ноутбук — короткая рабочая версия поверх одной SQLite-базы:

        - официальные ники и позиции из Liquipedia;
        - fantasy-очки по каждой карте;
        - role-category score: `core_avg`, `mid`, `support_avg`;
        - reliability-v2 score 1-100;
        - конструктор пользовательских fantasy-баннеров;
        - source-first агент, который не выдумывает внешние факты.

        Основной файл базы: `../data/ewc_2026_fantasy_compact.sqlite`.
        


In [35]:
from pathlib import Path
import importlib
import os
import sys
import sqlite3
import pandas as pd

# Layout switch:
# - "flat_colab"  : files uploaded into one flat /content directory
# - "project"     : normal repository structure with src/ and data/
NOTEBOOK_LAYOUT = "flat_colab"

# Optional manual overrides.
CUSTOM_PROJECT_ROOT = None
CUSTOM_SRC_DIR = None
CUSTOM_DB_PATH = None

# Ranking filter switch for notebook analytics.
# True  -> player/stat rankings only show teams qualified to TI 2026
# False -> include all teams present in the EWC database
TI_QUALIFIED_ONLY = True

DEFAULT_DB_FILENAME = "ewc_2026_fantasy_compact.sqlite"


def resolve_project_root() -> Path:
    if CUSTOM_PROJECT_ROOT:
        root = Path(CUSTOM_PROJECT_ROOT).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"CUSTOM_PROJECT_ROOT does not exist: {root}")
        return root

    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        Path("/content"),
        Path("/content/fantasy-analytics"),
        Path("/content/project"),
        Path("/content/drive/MyDrive/fantasy-analytics"),
    ])

    for candidate in candidates:
        src_ok = (candidate / "src" / "ewc_fact_agent_tools.py").exists()
        data_ok = (candidate / "data" / DEFAULT_DB_FILENAME).exists()
        if src_ok and data_ok:
            return candidate

    checked = "\n - ".join(str(p) for p in candidates)
    raise FileNotFoundError(
        "Could not resolve project layout automatically. "
        "Set CUSTOM_PROJECT_ROOT manually. Checked:\n - " + checked
    )


if NOTEBOOK_LAYOUT == "flat_colab":
    SRC_DIR = Path(CUSTOM_SRC_DIR or "/content").expanduser().resolve()
    DB_PATH = Path(CUSTOM_DB_PATH or f"/content/{DEFAULT_DB_FILENAME}").expanduser().resolve()
    PROJECT_ROOT = SRC_DIR
elif NOTEBOOK_LAYOUT == "project":
    PROJECT_ROOT = resolve_project_root()
    SRC_DIR = Path(CUSTOM_SRC_DIR).expanduser().resolve() if CUSTOM_SRC_DIR else (PROJECT_ROOT / "src").resolve()
    DB_PATH = Path(CUSTOM_DB_PATH).expanduser().resolve() if CUSTOM_DB_PATH else (PROJECT_ROOT / "data" / DEFAULT_DB_FILENAME).resolve()
else:
    raise ValueError("NOTEBOOK_LAYOUT must be 'flat_colab' or 'project'")

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database file not found: {DB_PATH}")
if not (SRC_DIR / "ewc_fact_agent_tools.py").exists():
    raise FileNotFoundError(f"ewc_fact_agent_tools.py not found in: {SRC_DIR}")

try:
    os.chmod(DB_PATH, 0o666)
except OSError:
    pass

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

importlib.invalidate_caches()
import ewc_fact_agent_tools


def patched_connect(db_path=None):
    con = sqlite3.connect(str(DB_PATH))
    con.row_factory = sqlite3.Row
    return con


ewc_fact_agent_tools.connect = patched_connect
ewc_fact_agent_tools.DB_PATH = DB_PATH

from ewc_fact_agent_tools import (
    EWCFactAgent,
    db_status,
    ask,
    roster,
    top_fantasy_maps,
    player_maps,
    role_map_summary,
    reliable_players_v2,
    reliable_role_slots_v2,
    reliability_backtest_v2,
    ti_qualified_teams,
    source_cache_status,
    banner_optimizer_players,
    banner_optimizer_role_slots,
    scoring_formula,
    source_urls,
    explain_sql_plan,
    explain_system_short,
)

agent = EWCFactAgent(DB_PATH)


def ask_v2(question: str, max_rows: int | None = None, use_llm: bool = False):
    """Main helper: returns AgentResult and prints markdown answer."""
    result = agent.ask(question, max_rows=max_rows, use_llm=use_llm)
    print(result.answer_markdown)
    return result

print("Clean EWC 2026 fact-agent ready.")
print(f"Layout mode: {NOTEBOOK_LAYOUT}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SRC_DIR: {SRC_DIR}")
print(f"DB_PATH: {DB_PATH}")
print(f"TI_QUALIFIED_ONLY: {TI_QUALIFIED_ONLY}")



Clean EWC 2026 fact-agent готов. Применен глубокий патч путей для Colab.


        ## 1. Быстрая проверка базы

        Если все ключевые объекты существуют, можно пользоваться агентом.
        


In [36]:
# Глубокая проверка подключения
import sqlite3
import os

try:
    # 1. Пробуем открыть файл средствами Python напрямую
    with open(DB_PATH, 'rb') as f:
        header = f.read(100)
        print(f"Файл читается как системный объект. Заголовок: {header[:15]}...")

    # 2. Пробуем открыть через sqlite3 с явным URI
    conn = sqlite3.connect(f"file:{DB_PATH}?mode=rw", uri=True)
    conn.execute("SELECT 1").fetchone()
    conn.close()
    print("Прямое подключение через sqlite3: Успешно!")

    # 3. Проверка через обертку агента
    status = db_status()
    display(status)
except Exception as e:
    print(f"Критическая ошибка: {e}")
    print(f"DB_PATH: {DB_PATH}")
    print(f"Доступные файлы: {os.listdir('/content')}")

Файл читается как системный объект. Заголовок: b'SQLite format 3'...
Прямое подключение через sqlite3: Успешно!


,object,exists,rows
0,matches,True,157
1,player_identity_registry,True,120
2,analytics_player_maps,True,1570
3,analytics_team_role_maps,True,314
4,dota_heroes,True,127
5,analytics_sources,True,10
6,analytics_ti2026_teams,True,16
7,analytics_reliable_players,True,120
8,analytics_reliable_role_slots,True,72
9,analytics_optimizer_players,True,114


        ## 2. Основной агент

        Агент сначала пытается решить вопрос через SQLite. Если в вопросе есть внешний фильтр вроде `TI 2026 qualification`, он не делает вид, что этот список есть в базе, а просит сверить источник.
        


In [37]:
examples = [
    "какой был состав у BetBoom?",
    "Укажи топ 15 лучших фентези игроков 1 позиции, их команды и лучшие фэнтези результаты",
    "Укажи топ 15 лучших фентези игроков 1 позиции из команд отобравшихся на TI 2026",
    "оптимизируй баннер для игроков 1 позиции из команд отобравшихся на TI 2026",
    "покажи надежных игроков для фэнтези",
    "покажи надежных саппортов для фэнтези",
    "самые надежные core пары",
    "покажи backtest модели надежности",
    "подробно объясни как считались fantasy очки",
]

for q in examples[:3]:
    print("\nQUESTION:", q)
    _ = ask_v2(q, max_rows=8)



QUESTION: какой был состав у BetBoom?
### Состав BoomBoys

```text
team_name  official_position role_group official_name            db_player_name  account_id                      source_name                                          source_url
 BoomBoys                  1       core     Kiritych~                   Naruto~   172099728 Liquipedia EWC 2026 participants https://liquipedia.net/dota2/Esports_World_Cup/2026
 BoomBoys                  2        mid          gpk~                       www   480412663 Liquipedia EWC 2026 participants https://liquipedia.net/dota2/Esports_World_Cup/2026
 BoomBoys                  3       core         MieRo                   17 mind   165564598 Liquipedia EWC 2026 participants https://liquipedia.net/dota2/Esports_World_Cup/2026
 BoomBoys                  4    support         Save- yo terase mugen tsukuyomi   317880638 Liquipedia EWC 2026 participants https://liquipedia.net/dota2/Esports_World_Cup/2026
 BoomBoys                  5    support      Ka

        ## 3. Прямые SQL-friendly helpers

        Эти функции удобны, когда не нужен natural language router.
        


In [38]:
# Принудительное использование глобального агента и патча путей
display(roster("Team Falcons"))
display(top_fantasy_maps(position=1, limit=10))
display(top_fantasy_maps(position=1, ti2026_only=True, limit=10))
display(reliable_players_v2(position=1, limit=10))
display(reliable_role_slots_v2(role_slot="core_pair", limit=10))
display(ti_qualified_teams())
display(reliability_backtest_v2())

,team_name,official_position,role_group,official_name,db_player_name,account_id,source_name,source_url
0,Team Falcons,1,core,skiter,I m Going To Be 10k Player,100058342,Liquipedia EWC 2026 participants,https://liquipedia.net/dota2/Esports_World_Cup...
1,Team Falcons,2,mid,Malr1ne,Буська,898455820,Liquipedia EWC 2026 participants,https://liquipedia.net/dota2/Esports_World_Cup...
2,Team Falcons,3,core,ATF,Sumaoky-,183719386,Liquipedia EWC 2026 participants,https://liquipedia.net/dota2/Esports_World_Cup...
3,Team Falcons,4,support,Cr1t-,o_o,25907144,Liquipedia EWC 2026 participants,https://liquipedia.net/dota2/Esports_World_Cup...
4,Team Falcons,5,support,Sneyking,Sneyking,10366616,Liquipedia EWC 2026 participants,https://liquipedia.net/dota2/Esports_World_Cup...


,fantasy_score,official_name,team_name,official_position,role_group,hero_name,match_id,match_date,stage_name,opponent_name,won,duration_sec,qualification_path,ti_region
0,21151.97,Ghost,GamerLegion,1,core,Nature's Prophet,8885250776,2026-07-07,Group Stage,_PowerRangers,1,2845.0,North America Qualifier,North America
1,19668.08,Crystallis,MOUZ,1,core,Monkey King,8892923329,2026-07-12,Group Stage,Vici Gaming,0,4388.0,None,None
2,18333.56,Timado,Virtus.pro,1,core,Windranger,8885859920,2026-07-07,Group Stage,LGD Gaming,1,4599.0,None,None
3,17474.03,Jikroy,REKONIX,1,core,Lone Druid,8892835652,2026-07-12,Group Stage,Team Nemesis,1,3884.0,None,None
4,17240.92,watson,Team Yandex,1,core,Phantom Lancer,8893165071,2026-07-12,Group Stage,1w,1,3619.0,Direct invite,Direct invite
5,17238.40,Ame,Xtreme Gaming,1,core,Clinkz,8896267691,2026-07-14,Survival Stage,Team Liquid,0,5633.0,Direct invite,Direct invite
6,17171.57,m1CKe,Team Liquid,1,core,Shadow Fiend,8896140998,2026-07-14,Survival Stage,Xtreme Gaming,0,4155.0,Direct invite,Direct invite
7,17163.55,shiro,Vici Gaming,1,core,Lina,8897971312,2026-07-15,Survival Stage,1w,1,2723.0,China Qualifier,China
8,17060.57,Timado,Virtus.pro,1,core,Gyrocopter,8893253595,2026-07-12,Group Stage,OG,1,3971.0,None,None
9,16892.02,Yuma,LGD Gaming,1,core,Necrophos,8886013461,2026-07-07,Group Stage,Virtus.pro,1,3696.0,South America Qualifier,South America


,fantasy_score,official_name,team_name,official_position,role_group,hero_name,match_id,match_date,stage_name,opponent_name,won,duration_sec,qualification_path,ti_region
0,21151.97,Ghost,GamerLegion,1,core,Nature's Prophet,8885250776,2026-07-07,Group Stage,_PowerRangers,1,2845.0,North America Qualifier,North America
1,17240.92,watson,Team Yandex,1,core,Phantom Lancer,8893165071,2026-07-12,Group Stage,1w,1,3619.0,Direct invite,Direct invite
2,17238.40,Ame,Xtreme Gaming,1,core,Clinkz,8896267691,2026-07-14,Survival Stage,Team Liquid,0,5633.0,Direct invite,Direct invite
3,17171.57,m1CKe,Team Liquid,1,core,Shadow Fiend,8896140998,2026-07-14,Survival Stage,Xtreme Gaming,0,4155.0,Direct invite,Direct invite
4,17163.55,shiro,Vici Gaming,1,core,Lina,8897971312,2026-07-15,Survival Stage,1w,1,2723.0,China Qualifier,China
5,16892.02,Yuma,LGD Gaming,1,core,Necrophos,8886013461,2026-07-07,Group Stage,Virtus.pro,1,3696.0,South America Qualifier,South America
6,16776.80,Kiritych~,BoomBoys,1,core,Drow Ranger,8904157676,2026-07-19,Playoffs,PVISION,0,3720.0,Direct invite,Direct invite
7,16637.70,Satanic,PVISION,1,core,Nature's Prophet,8902431810,2026-07-18,Playoffs,Team Yandex,0,3947.0,European Qualifier,Europe
8,16223.93,m1CKe,Team Liquid,1,core,Alchemist,8896267691,2026-07-14,Survival Stage,Xtreme Gaming,1,5633.0,Direct invite,Direct invite
9,16183.90,Ghost,GamerLegion,1,core,Necrophos,8886641793,2026-07-08,Group Stage,Xtreme Gaming,0,3901.0,North America Qualifier,North America


,reliability_score_1_100,official_name,team_name,official_position,role_group,predicted_score_raw,low_estimate,expected_estimate,high_estimate,uncertainty_score,confidence_label,train_best2_series_score,train_second_best2_series_score,repeatability_ratio,spike_gap,shrinkage_weight,uncertainty_penalty,train_series_seen,data_quality_label
0,100.00,Timado,Virtus.pro,1,core,24670.553187,20250.95,24670.55,29090.16,17.91,stable,30618.06,30551.57,0.997828,66.49,0.636364,421.046099,7,usable_for_default_recommendations
1,97.89,Ghost,GamerLegion,1,core,23383.566463,18552.15,23383.57,28214.98,20.66,medium_uncertainty,33613.47,27091.30,0.805966,6522.17,0.555556,629.741378,5,usable_for_default_recommendations
2,95.79,Crystallis,MOUZ,1,core,22565.543911,17740.51,22565.54,27390.58,21.38,medium_uncertainty,29914.61,26324.44,0.879986,3590.17,0.600000,554.300058,6,usable_for_default_recommendations
3,93.68,Yatoro,Team Spirit,1,core,22251.211116,18936.22,22251.21,25566.20,14.90,stable,28230.19,24464.79,0.866618,3765.40,0.600000,377.741013,6,usable_for_default_recommendations
4,91.57,Pure,1w,1,core,22214.818760,18637.71,22214.82,25791.92,16.10,stable,28678.59,24592.40,0.857518,4086.19,0.600000,420.494010,6,usable_for_default_recommendations
5,89.47,shiro,Vici Gaming,1,core,22202.402001,16472.48,22202.40,27932.32,25.81,medium_uncertainty,31961.15,25974.29,0.812683,5986.86,0.636364,788.389156,7,usable_for_default_recommendations
6,87.36,Ame,Xtreme Gaming,1,core,22094.841826,16875.20,22094.84,27314.49,23.62,medium_uncertainty,31946.63,24824.58,0.777064,7122.05,0.600000,721.932681,6,usable_for_default_recommendations
7,85.26,Natsumi,OG,1,core,21986.324716,18401.53,21986.32,25571.12,16.30,stable,26167.30,25043.85,0.957067,1123.45,0.555556,306.198169,5,usable_for_default_recommendations
8,78.94,Yuma,LGD Gaming,1,core,21512.946721,17621.62,21512.95,25404.27,18.09,medium_uncertainty,28595.25,24786.32,0.866799,3808.93,0.666667,504.512014,8,usable_for_default_recommendations
9,76.83,Darklord^,Rune Eaters,1,core,21428.307389,19843.31,21428.31,23013.31,7.40,stable,22922.79,22753.84,0.992630,168.95,0.636364,86.971299,7,usable_for_default_recommendations


,reliability_score_1_100,team_name,role_slot,player_names,predicted_score_raw,train_best2_series_score,low_estimate,expected_estimate,high_estimate,uncertainty_score,confidence_label,train_second_best2_series_score,repeatability_ratio,spike_gap,shrinkage_weight,uncertainty_penalty,train_series_seen,data_quality_label
0,100.00,Virtus.pro,core_pair,"Timado, SaberLight",22239.474895,25302.170,18604.71,22239.47,25874.24,16.34,stable,24948.485,0.986022,353.685,0.700000,332.192065,7,usable_for_default_recommendations
1,95.70,Xtreme Gaming,core_pair,"Ame, Xxs",22086.057176,27741.350,18094.57,22086.06,26077.55,18.07,medium_uncertainty,24049.395,0.866915,3691.955,0.666667,454.502780,6,usable_for_default_recommendations
2,91.39,Team Falcons,core_pair,"skiter, ATF",22018.180280,24260.245,19538.36,22018.18,24498.00,11.26,stable,23307.170,0.960715,953.075,0.625000,176.161185,5,usable_for_default_recommendations
3,87.09,GamerLegion,core_pair,"Ghost, Fayde",21895.918058,27698.875,18033.33,21895.92,25758.50,17.64,stable,23333.610,0.842403,4365.265,0.625000,450.104195,5,usable_for_default_recommendations
4,82.78,LGD Gaming,core_pair,"Yuma, Wisper",21854.879782,25591.740,18795.92,21854.88,24913.84,14.00,stable,24830.440,0.970252,761.300,0.727273,293.327958,8,usable_for_default_recommendations
5,78.48,MOUZ,core_pair,"Crystallis, BOOM",21759.526294,26811.745,18012.33,21759.53,25506.72,17.22,stable,23579.585,0.879450,3232.160,0.666667,412.193527,6,usable_for_default_recommendations
6,74.17,Rune Eaters,core_pair,"Darklord^, Malik",21755.786752,23344.295,19810.32,21755.79,23701.25,8.94,stable,22775.830,0.975649,568.465,0.700000,149.623037,7,usable_for_default_recommendations
7,69.87,Team Spirit,core_pair,"Yatoro, Collapse",21502.521576,25743.135,18460.47,21502.52,24544.57,14.15,stable,22460.890,0.872500,3282.245,0.666667,336.433172,6,usable_for_default_recommendations
8,65.57,1w,core_pair,"Pure, 33",21430.188236,24476.420,18890.20,21430.19,23970.17,11.85,stable,22172.265,0.905862,2304.155,0.666667,246.082305,6,usable_for_default_recommendations
9,61.26,OG,core_pair,"Natsumi, Raven",21025.400250,22536.270,18314.83,21025.40,23735.97,12.89,stable,22026.385,0.977375,509.885,0.625000,189.301917,5,usable_for_default_recommendations


,team_name,source_team_name,qualification_path,region,roster_text,has_ewc_player_data,source_url,secondary_source_url,checked_at_utc,confidence_label
0,Vici Gaming,Vici Gaming,China Qualifier,China,shiro; Xm; Bach; XinQ; y`,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
1,1w,1win Team,Direct invite,Direct invite,Pure; bzm; 33; Ari; Whitemon,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
2,Aurora Gaming,Aurora Gaming,Direct invite,Direct invite,Nightfall; Mikoto; Ws; Mira; kaori,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
3,BoomBoys,BoomBoys / BetBoom Team,Direct invite,Direct invite,Kiritych~; gpk~; MieRo; Save-; Kataomi,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
4,Team Falcons,Team Falcons,Direct invite,Direct invite,skiter; Malr1ne; ATF; Cr1t-; Sneyking,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
5,Team Liquid,Team Liquid,Direct invite,Direct invite,m1CKe; Nisha; Ace; Boxi; tOfu,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
6,Team Yandex,Team Yandex,Direct invite,Direct invite,watson; CHIRA_JUNIOR; DM; Saksa; Malady,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
7,Xtreme Gaming,Xtreme Gaming,Direct invite,Direct invite,Ame; NothingToSay; Xxs; fy; xNova,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
8,Nigma Galaxy,Nigma Galaxy,European Qualifier,Europe,SumaiL; lorenof; Davai; OmaR; GH,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
9,PVISION,TEAM VISION / PARIVISION,European Qualifier,Europe,Satanic; No[o]ne-; Noticed; 9Class; Dukalis,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06


,entity_type,segment_name,n_test,mae,rmse,spearman_corr,top5_overlap_rate,top10_overlap_rate
0,player,all,40,3189.964925,4040.703327,0.718011,0.0,0.3
1,player,core,16,4345.490113,5065.574905,-0.082353,0.0,0.6
2,player,mid,8,3846.902231,4804.055904,-0.571429,0.6,1.0
3,player,support,16,1705.971084,1902.283565,0.388235,0.6,0.6
4,player_temporal,all,120,3229.312851,4411.591492,0.827356,0.0,0.2
5,player_temporal,core,48,4804.239390,5984.725204,0.537017,0.0,0.4
6,player_temporal,mid,24,3492.731616,4291.836780,0.332174,0.2,0.5
7,player_temporal,support,48,1522.676929,1904.857341,0.193878,0.8,0.5
8,role_slot,all,24,3149.452104,3963.604783,0.583478,0.0,0.5
9,role_slot,core_pair,8,4141.934135,4552.712556,-0.309524,0.6,1.0


        ## 4. Fantasy banner optimizer

        Optimizer использует текущий fantasy-профиль и оценивает привлекательность пика по повторяемому потолку, а не по простой средней карте.

        По умолчанию саппорты исключены из рекомендаций, потому что их статистика в этой базе low-confidence.
        


In [39]:
# Оптимизация теперь должна работать с корректным DB_PATH
display(banner_optimizer_players(position=1, ti2026_only=True, limit=15))
display(banner_optimizer_role_slots(role_slot="core_pair", ti2026_only=True, limit=10))

# Natural language route:
_ = ask_v2("оптимизируй баннер для игроков 1 позиции из команд отобравшихся на TI 2026", max_rows=10)

,optimizer_score_1_100,official_name,team_name,official_position,role_group,predicted_score_raw,best2_series_score,second_best2_series_score,repeatability_ratio,spike_gap,train_series_seen,ti2026_qualified,qualification_path,ti_region,data_quality_label,recommendation_note
0,100.00,Ghost,GamerLegion,1,core,27813.766357,33613.47,27091.30,0.805966,6522.17,5,1,North America Qualifier,North America,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
1,96.33,Ame,Xtreme Gaming,1,core,25064.256520,31946.63,24824.58,0.777064,7122.05,6,1,Direct invite,Direct invite,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
2,92.67,shiro,Vici Gaming,1,core,24976.739438,31961.15,25974.29,0.812683,5986.86,7,1,China Qualifier,China,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
3,89.00,Pure,1w,1,core,24688.435572,28678.59,24592.40,0.857518,4086.19,6,1,Direct invite,Direct invite,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
4,85.33,Yatoro,Team Spirit,1,core,24639.235093,28230.19,24464.79,0.866618,3765.40,6,1,European Qualifier,Europe,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
5,81.67,Natsumi,OG,1,core,24165.722132,26167.30,25043.85,0.957067,1123.45,5,1,Southeast Asia Qualifier,Southeast Asia,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
6,74.33,m1CKe,Team Liquid,1,core,23548.155426,33395.50,22128.13,0.662608,11267.37,7,1,Direct invite,Direct invite,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
7,70.67,Yuma,LGD Gaming,1,core,23525.750211,28595.25,24786.32,0.866799,3808.93,8,1,South America Qualifier,South America,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
8,67.00,watson,Team Yandex,1,core,23459.490800,27942.47,23337.32,0.835192,4605.15,5,1,Direct invite,Direct invite,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
9,63.33,Nightfall,Aurora Gaming,1,core,23306.498039,26365.52,23442.67,0.889141,2922.85,6,1,Direct invite,Direct invite,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.


,optimizer_score_1_100,team_name,role_slot,player_names,predicted_score_raw,best2_series_score,second_best2_series_score,repeatability_ratio,spike_gap,train_series_seen,ti2026_qualified,qualification_path,ti_region,data_quality_label,recommendation_note
0,100.00,GamerLegion,core_pair,"Fayde, Ghost",23535.256938,27698.88,23333.61,0.842403,4365.27,5,1,North America Qualifier,North America,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
1,92.38,Xtreme Gaming,core_pair,"Xxs, Ame",23482.585127,27741.35,24049.39,0.866915,3691.96,6,1,Direct invite,Direct invite,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
2,84.77,Team Falcons,core_pair,"skiter, ATF",22958.158325,24260.25,23307.17,0.960714,953.08,5,1,Direct invite,Direct invite,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
3,77.15,LGD Gaming,core_pair,"Yuma, Wisper",22801.304978,25591.74,24830.44,0.970252,761.30,8,1,South America Qualifier,South America,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
4,69.54,Team Spirit,core_pair,"Collapse, Yatoro",22466.847360,25743.14,22460.89,0.872500,3282.25,6,1,European Qualifier,Europe,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
5,61.92,1w,core_pair,"33, Pure",22146.836898,24476.42,22172.27,0.905862,2304.15,6,1,Direct invite,Direct invite,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
6,54.31,Team Liquid,core_pair,"Ace, m1CKe",21604.670596,28714.93,20871.45,0.726850,7843.48,7,1,Direct invite,Direct invite,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
7,46.69,Vici Gaming,core_pair,"Bach, shiro",21409.958592,24279.86,22947.35,0.945119,1332.51,7,1,China Qualifier,China,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
8,39.08,PVISION,core_pair,"Noticed, Satanic",21330.087109,23647.49,21308.42,0.901086,2339.07,5,1,European Qualifier,Europe,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.
9,31.46,OG,core_pair,"Raven, Natsumi",21299.809307,22536.27,22026.38,0.977375,509.89,5,1,Southeast Asia Qualifier,Southeast Asia,usable_for_default_recommendations,Recommended by repeatable ceiling optimizer.


### Оптимизатор fantasy-игроков среди TI 2026 qualified

```text
 optimizer_score_1_100 official_name     team_name  official_position role_group  predicted_score_raw  best2_series_score  second_best2_series_score  repeatability_ratio  spike_gap  train_series_seen  ti2026_qualified       qualification_path      ti_region                 data_quality_label                          recommendation_note
                100.00         Ghost   GamerLegion                  1       core         27813.766357            33613.47                   27091.30             0.805966    6522.17                  5                 1  North America Qualifier  North America usable_for_default_recommendations Recommended by repeatable ceiling optimizer.
                 96.33           Ame Xtreme Gaming                  1       core         25064.256520            31946.63                   24824.58             0.777064    7122.05                  6                 1            Direct invite  Direct invite u

        ## 5. Конструктор fantasy-баннеров

        Можно создать профиль под любые выпавшие коэффициенты. Профиль сохранится в тех же таблицах:

        - `fantasy_scoring_profiles`;
        - `fantasy_scoring_profile_stats`;
        - `fantasy_scoring_profile_banners`;
        - `fantasy_player_map_scores`;
        - `fantasy_team_role_map_scores`;
        - `fantasy_pick_value`.

        Ячейка ниже безопасная: пример не запускается автоматически.
        


In [40]:
from fantasy_profile_constructor import create_or_replace_banner_profile

MY_CUSTOM_BANNER = {
    "core": [
        ("kills", 2.5),
        ("creep_score", 2.5),
        ("teamfight_participation", 1.8),
    ],
    "mid": [
        ("creep_score", 2.7),
        ("runes_grabbed", 1.8),
        ("teamfight_participation", 2.7),
    ],
    "support": [
        ("lotus", 3.2),
        ("watchers_taken", 2.1),
        ("teamfight_participation", 1.5),
    ],
}

# Чтобы реально создать профиль, поменяй False на True.
if False:
    con = sqlite3.connect(DB_PATH)
    profile_id = create_or_replace_banner_profile(
        con,
        "my_new_banner_profile",
        MY_CUSTOM_BANNER,
        profile_name="My new fantasy banner",
        set_default=False,  # True сделает профиль дефолтным
    )
    con.close()
    print("created:", profile_id)


        ## 6. Web/source tools

        Source-cache уже хранит OpenDota heroes и TI 2026 qualified teams. Для новых внешних фактов агент все равно не должен фантазировать: сначала источник, потом SQL-фильтр.
        


In [41]:
display(source_cache_status())
display(ti_qualified_teams())
display(source_urls("команды отобравшиеся на TI 2026"))

# Если среда разрешает интернет, можно вручную попробовать:
# from ewc_fact_agent_tools import fetch_url_text
# text = fetch_url_text("https://liquipedia.net/dota2/The_International/2026")
# print(text[:1000])


,source_key,source_name,source_url,fetched_at_utc,status,content_type,http_status,notes
0,opendota_heroes,OpenDota heroes API,https://api.opendota.com/api/heroes,2026-08-06T05:07:08+00:00,fetched,application/json; charset=utf-8,200.0,Hero id mapping used to fill hero_name from pl...
1,liquipedia_ti2026_participants,Liquipedia The International 2026,https://liquipedia.net/dota2/The_International...,2026-08-06T05:07:08+00:00,extracted,text/html,NaN,Primary source for TI 2026 participants; page ...
2,dotesports_ti2026_teams,Dot Esports TI 2026 teams article,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,extracted,text/html,NaN,Secondary cross-check source for TI 2026 parti...
3,legacy_source_page_dotabuff_overview,dotabuff_overview,https://www.dotabuff.com/esports/leagues/19785...,2026-08-05T00:00:00+00:00,verified_web_snapshot,None,NaN,Dotabuff league overview checked; public page ...
4,legacy_source_page_dotabuff_matches,dotabuff_matches,https://www.dotabuff.com/esports/leagues/19785...,2026-08-05T00:00:00+00:00,verified_web_snapshot,None,NaN,Dotabuff matches page reports 1-20 of 159. Dir...
5,legacy_source_page_dotabuff_teams,dotabuff_teams,https://www.dotabuff.com/esports/leagues/19785...,2026-08-05T00:00:00+00:00,verified_web_snapshot,None,NaN,Dotabuff teams page exposes aggregate KDA/Kill...
6,legacy_source_page_dotabuff_players,dotabuff_players,https://www.dotabuff.com/esports/leagues/19785...,2026-08-05T00:00:00+00:00,verified_web_snapshot,None,NaN,Dotabuff players page exposes aggregate player...
7,legacy_source_page_liquipedia,liquipedia,https://liquipedia.net/dota2/Esports_World_Cup...,2026-08-05T00:00:00+00:00,verified_web_snapshot,None,NaN,"Liquipedia page used for tournament format, te..."
8,legacy_source_page_battlepass_fantasy_rules,battlepass_fantasy_rules,https://fantasy.battlepass.ru/guide,2026-08-05 05:59:17,rules_used,None,NaN,Official BattlePass Fantasy guide scoring coef...
9,legacy_source_page_opendota_fantasy_api,opendota_fantasy_api,https://api.opendota.com/api/matches/{match_id},2026-08-05 05:45:37,imported_api_json,None,NaN,Used only for fantasy fields absent from local...


,team_name,source_team_name,qualification_path,region,roster_text,has_ewc_player_data,source_url,secondary_source_url,checked_at_utc,confidence_label
0,Vici Gaming,Vici Gaming,China Qualifier,China,shiro; Xm; Bach; XinQ; y`,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
1,1w,1win Team,Direct invite,Direct invite,Pure; bzm; 33; Ari; Whitemon,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
2,Aurora Gaming,Aurora Gaming,Direct invite,Direct invite,Nightfall; Mikoto; Ws; Mira; kaori,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
3,BoomBoys,BoomBoys / BetBoom Team,Direct invite,Direct invite,Kiritych~; gpk~; MieRo; Save-; Kataomi,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
4,Team Falcons,Team Falcons,Direct invite,Direct invite,skiter; Malr1ne; ATF; Cr1t-; Sneyking,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
5,Team Liquid,Team Liquid,Direct invite,Direct invite,m1CKe; Nisha; Ace; Boxi; tOfu,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
6,Team Yandex,Team Yandex,Direct invite,Direct invite,watson; CHIRA_JUNIOR; DM; Saksa; Malady,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
7,Xtreme Gaming,Xtreme Gaming,Direct invite,Direct invite,Ame; NothingToSay; Xxs; fy; xNova,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
8,Nigma Galaxy,Nigma Galaxy,European Qualifier,Europe,SumaiL; lorenof; Davai; OmaR; GH,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06
9,PVISION,TEAM VISION / PARIVISION,European Qualifier,Europe,Satanic; No[o]ne-; Noticed; 9Class; Dukalis,1,https://liquipedia.net/dota2/The_International...,https://dotesports.com/dota-2/news/dota-2-ti-2...,2026-08-06T05:07:08+00:00,cross_checked_likely_current_as_of_2026-08-06


,source,url,best_for
0,Liquipedia,https://liquipedia.net/dota2/Special:Search?se...,"rosters, tournament stages, participants, TI q..."
1,Dotabuff,https://www.dotabuff.com/esports/leagues/19785,"EWC 2026 match pages, player nick/position evi..."
2,OpenDota,https://api.opendota.com/api/explorer,raw match/player statistics if match_id is known


        ## 7. SQL planner and confidence intervals

        `explain_sql_plan(...)` shows which deterministic route, views, filters and SQL template the agent will use. This is the fastest way to debug complex questions before letting GigaChat polish the answer.

        Reliability-v2 rows now include interval columns: `low_estimate`, `expected_estimate`, `high_estimate`, `uncertainty_score`, `confidence_label`.
        


In [42]:
# Проверка SQL-плана и интервалов надежности
display(explain_sql_plan("top 15 fantasy pos1 players from TI 2026 qualified teams"))

cols = [
    "reliability_score_1_100",
    "official_name",
    "team_name",
    "official_position",
    "predicted_score_raw",
    "low_estimate",
    "expected_estimate",
    "high_estimate",
    "uncertainty_score",
    "confidence_label",
]
try:
    res = reliable_players_v2(position=1, ti2026_only=True, limit=10)
    display(res[cols])
except Exception as e:
    print(f"Ошибка отображения таблицы: {e}")

,key,value
0,route,top_fantasy_maps
1,intent,rank individual player-map fantasy scores
2,confidence,high
3,tables_or_views,"analytics_player_maps, fantasy_player_map_scor..."
4,filters,"{""limit"": 15, ""official_position"": 1, ""role_gr..."
5,metrics,fantasy_score
6,params,"[1, ""core""]"
7,missing_external_facts,
8,notes,
9,sql,"SELECT fantasy_score, official_name, team_name..."


,reliability_score_1_100,official_name,team_name,official_position,predicted_score_raw,low_estimate,expected_estimate,high_estimate,uncertainty_score,confidence_label
0,97.89,Ghost,GamerLegion,1,23383.566463,18552.15,23383.57,28214.98,20.66,medium_uncertainty
1,93.68,Yatoro,Team Spirit,1,22251.211116,18936.22,22251.21,25566.20,14.90,stable
2,91.57,Pure,1w,1,22214.818760,18637.71,22214.82,25791.92,16.10,stable
3,89.47,shiro,Vici Gaming,1,22202.402001,16472.48,22202.40,27932.32,25.81,medium_uncertainty
4,87.36,Ame,Xtreme Gaming,1,22094.841826,16875.20,22094.84,27314.49,23.62,medium_uncertainty
5,85.26,Natsumi,OG,1,21986.324716,18401.53,21986.32,25571.12,16.30,stable
6,78.94,Yuma,LGD Gaming,1,21512.946721,17621.62,21512.95,25404.27,18.09,medium_uncertainty
7,74.72,Nightfall,Aurora Gaming,1,21389.184183,18099.43,21389.18,24678.94,15.38,stable
8,72.62,watson,Team Yandex,1,21321.595933,16976.16,21321.60,25667.03,20.38,medium_uncertainty
9,70.51,Kiritych~,BoomBoys,1,21105.786200,18651.38,21105.79,23560.20,11.63,stable


        ## 8. Dashboard and regression tests

        Dashboard is intentionally kept as a separate file so the notebook stays compact.

        - Dashboard file: `../dashboard/app.py`
        - Tests file: `../tests/regression_tests.py`
        


In [43]:
# В Colab используем плоскую структуру /content/ для всех файлов
DASHBOARD_PATH = "/content/app.py"
TESTS_PATH = "/content/regression_tests.py"

print("Dashboard launch command:")
print(f"streamlit run {DASHBOARD_PATH}")

print("\nRegression tests launch command:")
print(f"{sys.executable} {TESTS_PATH}")

# Оригинальные относительные пути закомментированы:
# !python "../tests/regression_tests.py"

Dashboard launch command:
streamlit run /content/app.py

Regression tests launch command:
/usr/bin/python3 /content/regression_tests.py


        ## 9. Optional GigaChat post-processing

        По умолчанию ответы deterministic. Если в Colab/окружении есть `GIGACHAT_CREDENTIALS`, можно вызвать:

        ```python
        ask_v2("покажи надежных игроков для фэнтези", use_llm=True)
        ```

        LLM получает только черновик и таблицы, поэтому не должна добавлять новые числа вне данных.
        


In [44]:
# Интерактивный режим:
# chat(use_llm=False)

# В конце работы можно закрыть соединение:
# agent.close()


# Раздел тестирования

In [55]:
# ## 3.1 Рейтинг fantasy-показателей для core_pair при коэффициенте 1.0

import sqlite3
import pandas as pd

con = sqlite3.connect(DB_PATH)

sql = """
WITH core_pair_maps AS (
    SELECT
        f.match_id,
        f.team_name
    FROM player_game_fantasy_summary f
    JOIN player_identity_registry pir
      ON pir.account_id = f.account_id
     AND pir.team_name = f.team_name
    WHERE pir.official_position IN (1, 3)
    GROUP BY f.match_id, f.team_name
    HAVING COUNT(DISTINCT pir.official_position) = 2
),
core_pair_stat_points AS (
    SELECT
        c.match_id,
        c.team_name,
        sc.stat_name,
        COALESCE(sc.emblem_color, 'unknown') AS color_group,
        AVG(COALESCE(sp.base_points, 0.0)) AS core_pair_stat_points_x1
    FROM core_pair_maps c
    JOIN player_game_fantasy_summary f
      ON f.match_id = c.match_id
     AND f.team_name = c.team_name
    JOIN player_identity_registry pir
      ON pir.account_id = f.account_id
     AND pir.team_name = f.team_name
     AND pir.official_position IN (1, 3)
    JOIN fantasy_scoring_stat_catalog sc
      ON 1 = 1
    LEFT JOIN fantasy_player_map_stat_points sp
      ON sp.match_id = f.match_id
     AND sp.account_id = f.account_id
     AND sp.team_name = f.team_name
     AND sp.stat_name = sc.stat_name
    GROUP BY
        c.match_id,
        c.team_name,
        sc.stat_name,
        sc.emblem_color
),
ranked AS (
    SELECT
        stat_name,
        color_group,
        core_pair_stat_points_x1,
        ROW_NUMBER() OVER (
            PARTITION BY stat_name
            ORDER BY core_pair_stat_points_x1
        ) AS rn,
        COUNT(*) OVER (
            PARTITION BY stat_name
        ) AS cnt
    FROM core_pair_stat_points
)
SELECT
    stat_name,
    color_group,
    COUNT(*) AS core_pair_maps,
    ROUND(AVG(core_pair_stat_points_x1), 2) AS avg_fantasy_points_x1,
    ROUND(MAX(core_pair_stat_points_x1), 2) AS max_fantasy_points_x1,
    ROUND(
        AVG(
            CASE
                WHEN rn IN (
                    CAST(((cnt - 1) * 0.75) AS INTEGER) + 1,
                    CAST(((cnt - 1) * 0.75) AS INTEGER) + 2
                )
                THEN core_pair_stat_points_x1
            END
        ),
        2
    ) AS p75_fantasy_points_x1
FROM ranked
GROUP BY stat_name, color_group
ORDER BY avg_fantasy_points_x1 DESC, stat_name
"""

core_pair_stat_ranking = pd.read_sql_query(sql, con)

display(core_pair_stat_ranking)

for color in ["red", "blue", "green"]:
    print(f"\n{color.upper()}")
    display(
        core_pair_stat_ranking
        .query("color_group == @color")
        .reset_index(drop=True)
    )

con.close()


,stat_name,color_group,core_pair_maps,avg_fantasy_points_x1,max_fantasy_points_x1,p75_fantasy_points_x1
0,creep_score,red,314,1346.22,2946.00,1601.25
1,gpm,red,314,1302.33,1792.00,1472.00
2,teamfight_participation,green,314,1292.32,1866.55,1449.11
3,deaths,red,314,1143.61,1950.00,1560.00
4,kills,red,314,674.20,2247.00,909.50
5,camps_stacked,blue,314,0.00,0.00,0.00
6,courier_kills,green,314,0.00,0.00,0.00
7,first_blood,green,314,0.00,0.00,0.00
8,lotus,blue,314,0.00,0.00,0.00
9,roshan_kills,green,314,0.00,0.00,0.00



RED


,stat_name,color_group,core_pair_maps,avg_fantasy_points_x1,max_fantasy_points_x1,p75_fantasy_points_x1
0,creep_score,red,314,1346.22,2946.0,1601.25
1,gpm,red,314,1302.33,1792.0,1472.00
2,deaths,red,314,1143.61,1950.0,1560.00
3,kills,red,314,674.20,2247.0,909.50



BLUE


,stat_name,color_group,core_pair_maps,avg_fantasy_points_x1,max_fantasy_points_x1,p75_fantasy_points_x1
0,camps_stacked,blue,314,0.0,0.0,0.0
1,lotus,blue,314,0.0,0.0,0.0
2,runes_grabbed,blue,314,0.0,0.0,0.0
3,smokes_used,blue,314,0.0,0.0,0.0
4,wards_placed,blue,314,0.0,0.0,0.0
5,watchers_taken,blue,314,0.0,0.0,0.0



GREEN


,stat_name,color_group,core_pair_maps,avg_fantasy_points_x1,max_fantasy_points_x1,p75_fantasy_points_x1
0,teamfight_participation,green,314,1292.32,1866.55,1449.11
1,courier_kills,green,314,0.00,0.00,0.00
2,first_blood,green,314,0.00,0.00,0.00
3,roshan_kills,green,314,0.00,0.00,0.00
4,stuns,green,314,0.00,0.00,0.00
5,tormentor_kills,green,314,0.00,0.00,0.00


In [58]:
# ## 3.1 Рейтинг fantasy-показателей для всех ролей при коэффициенте 1.0

import sqlite3
import pandas as pd

def get_stat_ranking(positions, group_label):
    con = sqlite3.connect(DB_PATH)

    # Определяем условия фильтрации: для пар нужно наличие обоих игроков в матче, для мидера - один игрок
    pos_list = ", ".join(map(str, positions))
    count_check = f"HAVING COUNT(DISTINCT pir.official_position) = {len(positions)}"

    sql = f"""
    WITH target_maps AS (
        SELECT f.match_id, f.team_name
        FROM player_game_fantasy_summary f
        JOIN player_identity_registry pir ON pir.account_id = f.account_id AND pir.team_name = f.team_name
        WHERE pir.official_position IN ({pos_list})
        GROUP BY f.match_id, f.team_name
        {count_check}
    ),
    stat_points AS (
        SELECT
            c.match_id, c.team_name, sc.stat_name,
            COALESCE(sc.emblem_color, 'unknown') AS color_group,
            AVG(COALESCE(sp.base_points, 0.0)) AS avg_points_x1
        FROM target_maps c
        JOIN player_game_fantasy_summary f ON f.match_id = c.match_id AND f.team_name = c.team_name
        JOIN player_identity_registry pir ON pir.account_id = f.account_id AND pir.official_position IN ({pos_list})
        JOIN fantasy_scoring_stat_catalog sc ON 1 = 1
        LEFT JOIN fantasy_player_map_stat_points sp ON sp.match_id = f.match_id AND sp.account_id = f.account_id AND sp.stat_name = sc.stat_name
        GROUP BY c.match_id, c.team_name, sc.stat_name, sc.emblem_color
    ),
    ranked AS (
        SELECT
            stat_name, color_group, avg_points_x1,
            ROW_NUMBER() OVER (PARTITION BY stat_name ORDER BY avg_points_x1) AS rn,
            COUNT(*) OVER (PARTITION BY stat_name) AS cnt
        FROM stat_points
    )
    SELECT
        stat_name, color_group,
        ROUND(AVG(avg_points_x1), 2) AS avg_fantasy_points_x1,
        ROUND(MAX(avg_points_x1), 2) AS max_fantasy_points_x1,
        ROUND(AVG(CASE WHEN rn IN (CAST(((cnt - 1) * 0.75) AS INTEGER) + 1, CAST(((cnt - 1) * 0.75) AS INTEGER) + 2) THEN avg_points_x1 END), 2) AS p75_fantasy_points_x1
    FROM ranked
    GROUP BY stat_name, color_group
    ORDER BY p75_fantasy_points_x1 DESC
    """
    df = pd.read_sql_query(sql, con)
    con.close()
    return df

for label, pos in [("CORE PAIR (1+3)", [1, 3]), ("MID (2)", [2]), ("SUPPORT PAIR (4+5)", [4, 5])]:
    print(f"\n{'='*20} {label} {'='*20}")
    ranking = get_stat_ranking(pos, label)
    display(ranking.head(10))


==================== CORE PAIR (1+3) ====================


,stat_name,color_group,avg_fantasy_points_x1,max_fantasy_points_x1,p75_fantasy_points_x1
0,creep_score,red,1346.22,2946.00,1601.25
1,deaths,red,1143.61,1950.00,1560.00
2,gpm,red,1302.33,1792.00,1472.00
3,teamfight_participation,green,1292.32,1866.55,1449.11
4,kills,red,674.20,2247.00,909.50
5,camps_stacked,blue,0.00,0.00,0.00
6,courier_kills,green,0.00,0.00,0.00
7,first_blood,green,0.00,0.00,0.00
8,lotus,blue,0.00,0.00,0.00
9,roshan_kills,green,0.00,0.00,0.00



==================== MID (2) ====================


,stat_name,color_group,avg_fantasy_points_x1,max_fantasy_points_x1,p75_fantasy_points_x1
0,teamfight_participation,green,1462.30,2124.0,1668.86
1,deaths,red,1171.86,1950.0,1560.00
2,creep_score,red,1238.24,4071.0,1483.50
3,gpm,red,1237.89,2142.0,1382.00
4,kills,red,794.32,2568.0,1070.00
5,camps_stacked,blue,0.00,0.0,0.00
6,courier_kills,green,0.00,0.0,0.00
7,first_blood,green,0.00,0.0,0.00
8,lotus,blue,0.00,0.0,0.00
9,roshan_kills,green,0.00,0.0,0.00



==================== SUPPORT PAIR (4+5) ====================


,stat_name,color_group,avg_fantasy_points_x1,max_fantasy_points_x1,p75_fantasy_points_x1
0,teamfight_participation,green,1403.43,2124.0,1522.97
1,deaths,red,598.35,1950.0,1072.50
2,gpm,red,663.11,1041.0,749.50
3,kills,red,305.84,909.5,428.00
4,creep_score,red,242.14,1263.0,305.25
5,camps_stacked,blue,0.00,0.0,0.00
6,courier_kills,green,0.00,0.0,0.00
7,first_blood,green,0.00,0.0,0.00
8,lotus,blue,0.00,0.0,0.00
9,roshan_kills,green,0.00,0.0,0.00


In [ ]:
# ## 3.2 ???????????? ??????? ?? ?????? ?????????? ??? role-????? core_pair / mid / support_pair

import sqlite3
import pandas as pd

ROLE_RANKING_CONFIG = {
    "core_pair": {
        "label": "CORE PAIR COMBINATIONS (positions 1 and 3)",
        "positions": [1, 3],
    },
    "mid": {
        "label": "MID OPTIONS (position 2)",
        "positions": [2],
    },
    "support_pair": {
        "label": "SUPPORT PAIR COMBINATIONS (positions 4 and 5)",
        "positions": [4, 5],
    },
}


def _build_role_slot_context(positions, ti_qualified_only=TI_QUALIFIED_ONLY):
    pos_list = ", ".join(map(str, positions))
    expected_positions = len(positions)
    team_filter_sql = """
        AND EXISTS (
            SELECT 1
            FROM analytics_ti2026_teams ti
            WHERE ti.team_name = pir.team_name
        )
    """ if ti_qualified_only else ""
    return pos_list, expected_positions, team_filter_sql


def get_role_slot_stat_ranking(positions, role_key, min_maps=1, ti_qualified_only=TI_QUALIFIED_ONLY):
    con = sqlite3.connect(DB_PATH)
    pos_list, expected_positions, team_filter_sql = _build_role_slot_context(
        positions,
        ti_qualified_only=ti_qualified_only,
    )
    sql = f"""
    WITH role_players AS (
        SELECT
            pir.account_id,
            pir.team_name,
            pir.official_name,
            pir.official_position,
            pir.role_group
        FROM player_identity_registry pir
        WHERE pir.official_position IN ({pos_list})
        {team_filter_sql}
    ),
    role_names AS (
        SELECT
            rp.team_name,
            GROUP_CONCAT(rp.official_name, ', ') AS player_names,
            MIN(rp.role_group) AS role_group
        FROM (
            SELECT team_name, official_name, official_position, role_group
            FROM role_players
            ORDER BY team_name, official_position
        ) rp
        GROUP BY rp.team_name
        HAVING COUNT(DISTINCT rp.official_position) = {expected_positions}
    ),
    target_maps AS (
        SELECT
            f.match_id,
            f.team_name
        FROM player_game_fantasy_summary f
        JOIN role_players rp
          ON rp.account_id = f.account_id
         AND rp.team_name = f.team_name
        GROUP BY f.match_id, f.team_name
        HAVING COUNT(DISTINCT rp.official_position) = {expected_positions}
    ),
    role_stat_maps AS (
        SELECT
            tm.match_id,
            tm.team_name,
            '{role_key}' AS role_slot,
            sc.stat_name,
            COALESCE(sc.emblem_color, 'unknown') AS color_group,
            AVG(COALESCE(sp.base_points, 0.0)) AS points_x1
        FROM target_maps tm
        JOIN role_players rp
          ON rp.team_name = tm.team_name
        JOIN fantasy_scoring_stat_catalog sc
          ON 1 = 1
        LEFT JOIN fantasy_player_map_stat_points sp
          ON sp.match_id = tm.match_id
         AND sp.account_id = rp.account_id
         AND sp.team_name = rp.team_name
         AND sp.stat_name = sc.stat_name
        GROUP BY tm.match_id, tm.team_name, sc.stat_name, sc.emblem_color
    ),
    ranked AS (
        SELECT
            rsm.team_name,
            rsm.role_slot,
            rn.player_names,
            rn.role_group,
            rsm.stat_name,
            rsm.color_group,
            rsm.points_x1,
            ROW_NUMBER() OVER (
                PARTITION BY rsm.team_name, rsm.role_slot, rsm.stat_name
                ORDER BY rsm.points_x1
            ) AS rn,
            COUNT(*) OVER (
                PARTITION BY rsm.team_name, rsm.role_slot, rsm.stat_name
            ) AS cnt
        FROM role_stat_maps rsm
        JOIN role_names rn
          ON rn.team_name = rsm.team_name
    )
    SELECT
        team_name,
        role_slot,
        player_names,
        role_group,
        stat_name,
        color_group,
        COUNT(*) AS maps_played,
        ROUND(AVG(points_x1), 2) AS avg_fantasy_points_x1,
        ROUND(MAX(points_x1), 2) AS max_fantasy_points_x1,
        ROUND(
            AVG(
                CASE
                    WHEN rn IN (
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 1,
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 2
                    )
                    THEN points_x1
                END
            ),
            2
        ) AS p75_fantasy_points_x1
    FROM ranked
    GROUP BY
        team_name,
        role_slot,
        player_names,
        role_group,
        stat_name,
        color_group
    HAVING COUNT(*) >= {int(min_maps)}
    ORDER BY stat_name, p75_fantasy_points_x1 DESC, max_fantasy_points_x1 DESC, team_name
    """
    df = pd.read_sql_query(sql, con)
    con.close()
    return df


def show_role_slot_stat_rankings(role_key, top_n=10, min_maps=1, ti_qualified_only=TI_QUALIFIED_ONLY):
    cfg = ROLE_RANKING_CONFIG[role_key]
    df = get_role_slot_stat_ranking(
        cfg["positions"],
        role_key=role_key,
        min_maps=min_maps,
        ti_qualified_only=ti_qualified_only,
    )
    print()
    print("=" * 24, cfg["label"], "=" * 24)
    print(f"Rows: {len(df)}")
    print(f"TI-qualified only: {ti_qualified_only}")
    for stat_name in df["stat_name"].drop_duplicates().tolist():
        chunk = (
            df[df["stat_name"] == stat_name]
            .sort_values(["p75_fantasy_points_x1", "max_fantasy_points_x1", "avg_fantasy_points_x1"], ascending=False)
            .head(top_n)
            .reset_index(drop=True)
        )
        print()
        print(f"STAT: {stat_name}")
        display(chunk)
    return df


role_slot_stat_rankings = {
    role_key: get_role_slot_stat_ranking(
        cfg["positions"],
        role_key=role_key,
        ti_qualified_only=TI_QUALIFIED_ONLY,
    )
    for role_key, cfg in ROLE_RANKING_CONFIG.items()
}
player_stat_rankings = role_slot_stat_rankings

for role_key in ROLE_RANKING_CONFIG:
    show_role_slot_stat_rankings(role_key, top_n=10, ti_qualified_only=TI_QUALIFIED_ONLY)



In [ ]:
# ## 3.3 ???????????? ??????? ?? ????? ??? ????? ???????? ?????? ?????? ? ?????????????

import sqlite3
import pandas as pd

MY_ROLE_BANNER_STATS = {
    "core_pair": MY_CUSTOM_BANNER["core"],
    "mid": MY_CUSTOM_BANNER["mid"],
    "support_pair": MY_CUSTOM_BANNER["support"],
}


def get_weighted_role_slot_ranking(positions, role_key, stat_weights, min_maps=1, ti_qualified_only=TI_QUALIFIED_ONLY):
    con = sqlite3.connect(DB_PATH)
    pos_list, expected_positions, team_filter_sql = _build_role_slot_context(
        positions,
        ti_qualified_only=ti_qualified_only,
    )
    values_sql = ",\n        ".join(
        f"('{stat_name}', {float(weight)})" for stat_name, weight in stat_weights
    )
    sql = f"""
    WITH selected_stats(stat_name, weight) AS (
        VALUES
        {values_sql}
    ),
    role_players AS (
        SELECT
            pir.account_id,
            pir.team_name,
            pir.official_name,
            pir.official_position,
            pir.role_group
        FROM player_identity_registry pir
        WHERE pir.official_position IN ({pos_list})
        {team_filter_sql}
    ),
    role_names AS (
        SELECT
            rp.team_name,
            GROUP_CONCAT(rp.official_name, ', ') AS player_names,
            MIN(rp.role_group) AS role_group
        FROM (
            SELECT team_name, official_name, official_position, role_group
            FROM role_players
            ORDER BY team_name, official_position
        ) rp
        GROUP BY rp.team_name
        HAVING COUNT(DISTINCT rp.official_position) = {expected_positions}
    ),
    target_maps AS (
        SELECT
            f.match_id,
            f.team_name
        FROM player_game_fantasy_summary f
        JOIN role_players rp
          ON rp.account_id = f.account_id
         AND rp.team_name = f.team_name
        GROUP BY f.match_id, f.team_name
        HAVING COUNT(DISTINCT rp.official_position) = {expected_positions}
    ),
    player_map_scores AS (
        SELECT
            tm.match_id,
            tm.team_name,
            rp.account_id,
            SUM(COALESCE(sp.base_points, 0.0) * ss.weight) AS weighted_points
        FROM target_maps tm
        JOIN role_players rp
          ON rp.team_name = tm.team_name
        JOIN selected_stats ss
          ON 1 = 1
        LEFT JOIN fantasy_player_map_stat_points sp
          ON sp.match_id = tm.match_id
         AND sp.account_id = rp.account_id
         AND sp.team_name = rp.team_name
         AND sp.stat_name = ss.stat_name
        GROUP BY tm.match_id, tm.team_name, rp.account_id
    ),
    role_map_scores AS (
        SELECT
            pms.match_id,
            pms.team_name,
            '{role_key}' AS role_slot,
            AVG(pms.weighted_points) AS weighted_points
        FROM player_map_scores pms
        GROUP BY pms.match_id, pms.team_name
    ),
    ranked AS (
        SELECT
            rms.team_name,
            rms.role_slot,
            rn.player_names,
            rn.role_group,
            rms.weighted_points,
            ROW_NUMBER() OVER (
                PARTITION BY rms.team_name, rms.role_slot
                ORDER BY rms.weighted_points
            ) AS rn,
            COUNT(*) OVER (
                PARTITION BY rms.team_name, rms.role_slot
            ) AS cnt
        FROM role_map_scores rms
        JOIN role_names rn
          ON rn.team_name = rms.team_name
    )
    SELECT
        team_name,
        role_slot,
        player_names,
        role_group,
        COUNT(*) AS maps_played,
        ROUND(AVG(weighted_points), 2) AS avg_weighted_points,
        ROUND(MAX(weighted_points), 2) AS max_weighted_points,
        ROUND(
            AVG(
                CASE
                    WHEN rn IN (
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 1,
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 2
                    )
                    THEN weighted_points
                END
            ),
            2
        ) AS p75_weighted_points
    FROM ranked
    GROUP BY team_name, role_slot, player_names, role_group
    HAVING COUNT(*) >= {int(min_maps)}
    ORDER BY p75_weighted_points DESC, max_weighted_points DESC, avg_weighted_points DESC, team_name
    """
    df = pd.read_sql_query(sql, con)
    con.close()
    return df


my_banner_role_rankings = {}
my_banner_player_rankings = my_banner_role_rankings
for role_key, stat_weights in MY_ROLE_BANNER_STATS.items():
    cfg = ROLE_RANKING_CONFIG[role_key]
    print()
    print("=" * 24, f"MY BANNER - {cfg['label']}", "=" * 24)
    print("Selected stats:", stat_weights)
    print(f"TI-qualified only: {TI_QUALIFIED_ONLY}")
    ranking_df = get_weighted_role_slot_ranking(
        cfg["positions"],
        role_key=role_key,
        stat_weights=stat_weights,
        ti_qualified_only=TI_QUALIFIED_ONLY,
    )
    my_banner_role_rankings[role_key] = ranking_df
    display(ranking_df.head(15))



In [ ]:
# ## 3.4 ??????????? ????? ?? ????? ? ???????????? ??????? ??? ??? ?????

import sqlite3
import pandas as pd

# Assumption for optimal slot templates:
# - core_pair    -> 2 red + 1 green
# - mid          -> 1 red + 1 blue + 1 green
# - support_pair -> 2 blue + 1 green
OPTIMAL_COLOR_TEMPLATES = {
    "core_pair": {"red": 2, "green": 1},
    "mid": {"red": 1, "blue": 1, "green": 1},
    "support_pair": {"blue": 2, "green": 1},
}


def get_role_stat_summary(positions, ti_qualified_only=TI_QUALIFIED_ONLY):
    con = sqlite3.connect(DB_PATH)
    pos_list = ", ".join(map(str, positions))
    count_check = f"HAVING COUNT(DISTINCT pir.official_position) = {len(positions)}"
    team_filter_sql = """
        AND EXISTS (
            SELECT 1
            FROM analytics_ti2026_teams ti
            WHERE ti.team_name = pir.team_name
        )
    """ if ti_qualified_only else ""
    sql = f"""
    WITH target_maps AS (
        SELECT
            f.match_id,
            f.team_name
        FROM player_game_fantasy_summary f
        JOIN player_identity_registry pir
          ON pir.account_id = f.account_id
         AND pir.team_name = f.team_name
        WHERE pir.official_position IN ({pos_list})
        {team_filter_sql}
        GROUP BY f.match_id, f.team_name
        {count_check}
    ),
    stat_points AS (
        SELECT
            c.match_id,
            c.team_name,
            sc.stat_name,
            COALESCE(sc.emblem_color, 'unknown') AS color_group,
            AVG(COALESCE(sp.base_points, 0.0)) AS avg_points_x1
        FROM target_maps c
        JOIN player_game_fantasy_summary f
          ON f.match_id = c.match_id
         AND f.team_name = c.team_name
        JOIN player_identity_registry pir
          ON pir.account_id = f.account_id
         AND pir.team_name = f.team_name
         AND pir.official_position IN ({pos_list})
        JOIN fantasy_scoring_stat_catalog sc
          ON 1 = 1
        LEFT JOIN fantasy_player_map_stat_points sp
          ON sp.match_id = f.match_id
         AND sp.account_id = f.account_id
         AND sp.team_name = f.team_name
         AND sp.stat_name = sc.stat_name
        GROUP BY c.match_id, c.team_name, sc.stat_name, sc.emblem_color
    ),
    ranked AS (
        SELECT
            stat_name,
            color_group,
            avg_points_x1,
            ROW_NUMBER() OVER (PARTITION BY stat_name ORDER BY avg_points_x1) AS rn,
            COUNT(*) OVER (PARTITION BY stat_name) AS cnt
        FROM stat_points
    )
    SELECT
        stat_name,
        color_group,
        COUNT(*) AS role_maps,
        ROUND(AVG(avg_points_x1), 2) AS avg_fantasy_points_x1,
        ROUND(MAX(avg_points_x1), 2) AS max_fantasy_points_x1,
        ROUND(
            AVG(
                CASE
                    WHEN rn IN (
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 1,
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 2
                    )
                    THEN avg_points_x1
                END
            ),
            2
        ) AS p75_fantasy_points_x1
    FROM ranked
    GROUP BY stat_name, color_group
    ORDER BY p75_fantasy_points_x1 DESC, max_fantasy_points_x1 DESC
    """
    df = pd.read_sql_query(sql, con)
    con.close()
    return df


def select_optimal_stats(role_key, ti_qualified_only=TI_QUALIFIED_ONLY):
    cfg = ROLE_RANKING_CONFIG[role_key]
    summary = get_role_stat_summary(
        cfg["positions"],
        ti_qualified_only=ti_qualified_only,
    )
    template = OPTIMAL_COLOR_TEMPLATES[role_key]
    selected = []
    for color_group, take_n in template.items():
        chunk = (
            summary[(summary["color_group"] == color_group) & (summary["p75_fantasy_points_x1"] > 0)]
            .sort_values(["p75_fantasy_points_x1", "max_fantasy_points_x1", "avg_fantasy_points_x1"], ascending=False)
            .head(take_n)
            .copy()
        )
        chunk["selected_weight"] = 1.0
        selected.append(chunk)
    selected_df = pd.concat(selected, ignore_index=True) if selected else pd.DataFrame()
    return summary, selected_df


optimal_role_stat_summaries = {}
optimal_role_selected_stats = {}
optimal_role_combo_rankings = {}
optimal_role_player_rankings = optimal_role_combo_rankings

for role_key in ROLE_RANKING_CONFIG:
    cfg = ROLE_RANKING_CONFIG[role_key]
    summary_df, selected_df = select_optimal_stats(
        role_key,
        ti_qualified_only=TI_QUALIFIED_ONLY,
    )
    optimal_role_stat_summaries[role_key] = summary_df
    optimal_role_selected_stats[role_key] = selected_df

    print()
    print("=" * 24, f"OPTIMAL STATS - {cfg['label']}", "=" * 24)
    print("Color template:", OPTIMAL_COLOR_TEMPLATES[role_key])
    print(f"TI-qualified only: {TI_QUALIFIED_ONLY}")
    display(selected_df.reset_index(drop=True))

    stat_weights = [(row.stat_name, 1.0) for row in selected_df.itertuples()]
    ranking_df = (
        get_weighted_role_slot_ranking(
            cfg["positions"],
            role_key=role_key,
            stat_weights=stat_weights,
            ti_qualified_only=TI_QUALIFIED_ONLY,
        )
        if stat_weights else pd.DataFrame()
    )
    optimal_role_combo_rankings[role_key] = ranking_df

    print()
    print(f"COMBINATIONS FOR OPTIMAL STATS - {cfg['label']}")
    display(ranking_df.head(15))

